<a href="https://colab.research.google.com/github/susanavenda/data_cambridge/blob/main/CAM_DS_301_Hugging_Face_prompts_Demo_3_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**First things first** - please go to 'File' and select 'Save a copy in Drive' so that you have your own version of this activity set up and ready to use.
Remember to update the portfolio index link to your own work once completed!

# Demonstration 3.1.1 Hugging Face prompts

In this demonstration, we will learn more about user prompts, which are useful and easy ways to use large language models (LLMs) and adapt them to a multitude of tasks. We will investigate how to specify system and user prompts on LLMs from Hugging Face to perform tasks such as text summarisation, question-answering, and machine translation.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 93.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = 'HuggingFaceH4/zephyr-7b-beta'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline
pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=100,
)

## Text summarisation

In [ ]:
torch.manual_seed(3)
prompt = """
<|system|>
You are a helpful langauge model designed to answer question faithfully
</s>
<|user|>
Generate a summary for the text provided
Text:
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.

</s>
<|assistant|>

 """


sequences = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)


In [ ]:

for seq in sequences:
    print(f"{seq['generated_text']}")

 The text highlights significant changes in America's educational landscape, particularly in engineering programs. Traditional disciplines like mechanical, civil, electrical, chemical, and aeronautical engineering have seen a decrease in graduates, and many top universities now prioritize the study of engineering science over infrastructure and environmental issues. This shift towards high-tech subjects supports scientific advancements but comes at the cost of traditional engineering education. In contrast, rapidly developing economies like China and India continue to promote engineering education,


## Question-answering

In [ ]:


prompt = """
<|system|>
You are a helpful langauge model designed to answer question faithfully
</s>
<|user|>
Answer the question using the context below.
Context: The Caesar salad, made with romaine lettuce and croutons, originated from Italian-American chef Caesar Cardini in the 1920s
Question: Who created the Caesar salad?
Answer:
</s>
<|assistant|>

 """

sequences = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Result:  The Caesar salad was created by Italian-American chef Caesar Cardini in the 1920s.


In [ ]:

prompt = """
<|system|>
You are a helpful langauge model designed to answer question faithfully
</s>
<|user|>
What is the capital city of  France?
Answer:
</s>
<|assistant|>

 """

sequences = pipe(
    prompt,
    max_new_tokens=10,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)


In [ ]:
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Result:  The capital city of France is Paris (French


## Machine Translation

In [ ]:

prompt = """
<|system|>
You are a helpful langauge model designed to answer question faithfully
</s>
<|user|>
Translate the following text to French
Text: I like data science
Translation:
</s>
<|assistant|>

 """

sequences = pipe(
    prompt,
    max_new_tokens=10,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)



In [ ]:
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Result:  J'aime la science des données



## Consider other models

Consider other models such as the following:

* [Mistral](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2)
* [LLAMA](https://huggingface.co/meta-llama/Meta-Llama-3-8B)

In [ ]:
# The mistral model invoked below can be accessed only with a hugging face pro account.

from huggingface_hub import login

login(token="add your access token here")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline
pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=100,
)

In [ ]:
prompt = """<s>[INST] <<SYS>>
You are a helpful langauge model designed to answer question faithfully

<</SYS>>
Generate a summary for the text provided
Text:
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
    Summary:

 [/INST]
"""


sequences = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Result: The text discusses the shift in engineering education in America towards engineering science, leading to a decline in traditional engineering disciplines such as mechanical, civil, electrical, chemical, and aeronautical engineering. This trend is contrasted with the continued emphasis on engineering education in rapidly developing economies like China and India, which graduate significantly more traditional engineers than the US. The author expresses concern that this decline could have negative consequences for infrastructure and environmental issues, and calls for a balance between high technology subjects


In [ ]:


prompt = """<s>[INST] <<SYS>>
You are a helpful langauge model designed to answer question faithfully

<</SYS>>
Translate the following text to French
Text: I like data science
Translation:

 [/INST]
"""

sequences = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Result: Je aim le data science (In English: "I like data science")

The translation in French for "I like" is "Je aim" and the term for "data science" remains unchanged. Therefore, the complete translation would be "Je aim le data science".


## Key information

In this demonstration, we explored user prompts and how they can be used within LLMs to, for example, summarise text, answer questions, and conduct machine translation.

## Reflect
What are the practical applications of these techniques?
> Select the pen from the toolbar to add your entry.